
# AUFS–DyClust Quickstart (Generic, Susenas-like)

This notebook shows how to run the **AUFS–DyClust (Engine C + auto‑K)** pipeline on a ready-to-cluster mixed‑type dataset.

**Input demo**: `../data/household_clustering_ready.csv` (semicolon `;` separated)

Expected columns include a household identifier (e.g., `HHID`) and mixed features such as `PlantProteinShare`, `FVShare`, `ProcessedShare`, `EnergyBurden`, categorical access variables, etc.


## import os, sys
from pathlib import Path

# asumsi notebook berada di notebooks/
ROOT = Path("..").resolve()
SRC = ROOT / "src"
sys.path.insert(0, str(SRC))

print(f"Project root  : {ROOT}")
print(f"Added to path : {SRC}")


In [1]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
SRC = ROOT / "src"
sys.path.insert(0, str(SRC))

print("SRC:", SRC)
print("Exists:", SRC.exists())
print("Contents:", list(SRC.iterdir()) if SRC.exists() else "NOT FOUND")
print("mixclust?:", (SRC / "mixclust").exists())

SRC: /home/jupyter-33220015/dynamic clustering/src
Exists: True
Contents: [PosixPath('/home/jupyter-33220015/dynamic clustering/src/.ipynb_checkpoints'), PosixPath('/home/jupyter-33220015/dynamic clustering/src/mixclust')]
mixclust?: True


In [2]:
# cell 3 - load data (robust to separator)
import pandas as pd
from pathlib import Path

DATA_PATH = Path("../data/obesity.csv")
LABEL = "NObeyesdad"   # nama kolom label di dataset UCI; ganti sesuai dataset
IDCOL = None                     # set None bila tidak ada ID kolom
DROPCOL = None #Untuk list ['fnlwgt','col_lain'] isi None jika tidak ada
ANCHORVAR=['Weight', 'BMI_category', 'family_history_with_overweight']
# detect sep automatically if you want, but here we assume comma for UCI
df = pd.read_csv(DATA_PATH, sep=",")
print("Loaded rows:", len(df))
df.head(3)


Loaded rows: 2111


,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,Female,21.0,1.62,64.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,0.0,1.0,no,Public_Transportation,Normal_Weight
1,Female,21.0,1.52,56.0,yes,no,3.0,3.0,Sometimes,yes,3.0,yes,3.0,0.0,Sometimes,Public_Transportation,Normal_Weight
2,Male,23.0,1.80,77.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,2.0,1.0,Frequently,Public_Transportation,Normal_Weight



## 1) Minimal run — Engine C + auto‑K

This uses the generic end‑to‑end runner, which internally calls `run_aufs_samba` (Phase A feature selection + Phase B auto‑K) and writes useful artifacts (selected features, cluster assignments, profiles, metrics).


In [3]:
# cell 5 - run pipeline
from datetime import datetime
from mixclust.pipeline.generic_end2end import run_generic_end2end
from mixclust.aufs_samba.api import AUFSParams
from pathlib import Path

run_id = datetime.now().strftime("%Y%m%d-%H%M%S")
outdir = Path(f"runs/demo-{run_id}")    # will be created by pipeline if needed

# params (tweak as needed)
params = AUFSParams(
    engine_mode="C",
    auto_k=True, c_min=3, c_max=7,        # ← 3-5 sesuai ekspektasi
    reward_metric="lsil_fixed_calibrated",
    lsil_topk=1, per_cluster_proto_if_many=1, lsil_proto_sample_cap=200,
    guard_every_calib=50, ss_max_n_cal=200,
    reward_subsample_n=20000, calibrate_mode="topk", calib_cache_enabled=True,
    ss_max_n=2000,
    enable_rerank=False, rerank_mode="none",
    rerank_topk=15,
    shadow_rerank=False,
    kmsnc_k=5, build_redundancy_parallel=True,
    build_redundancy_cache="cache/redundancy_k5.pkl",
    mab_n_jobs=2, red_backend="loky",
    sa_neighbor_mode="full", sa_min_size=3, sa_max_size=None,
    mab_T=12, mab_k=6,
    sa_iters=50,
    hac_mode="hybrid",
    cluster_adapter_lambda=0.6,
    enable_screening=True,
    screening_k_values=(3, 4, 5,6,7),          # ← sesuai c_min/c_max
    screening_prune_threshold=0.20,
    auto_algorithms=["kprototypes", "hac_gower"],
    random_state=42, verbose=True, show_progress=True,
    # Aktifkan DAV — cukup isi anchor cols
    dav_anchor_cols=ANCHORVAR
)

# Prepare df_ready: remove label column from features (but keep original df for later join)
#drop_cols = LABEL if LABEL in df.columns else None
drop_cols_list = []
if LABEL in df.columns:
    drop_cols_list.append(LABEL)

# Tambahkan fnlwgt ke drop list
if DROPCOL is not None:
    if isinstance(DROPCOL, str):
        if DROPCOL in df.columns:
            drop_cols_list.append(DROPCOL)
    elif isinstance(DROPCOL, list):
        for col in DROPCOL:
            if col in df.columns:
                drop_cols_list.append(col)
    
df_ready = df.drop(columns=drop_cols_list, errors="ignore").copy()

res = run_generic_end2end(
    df_ready=df_ready,
    outdir=str(outdir),
    id_col=IDCOL,         # None for UCI without ID
    drop_cols=drop_cols_list,  # pipeline will ignore these columns
    params=params,
    n_clusters_hint=3,
    verbose=True
)
res


[ADAPTIVE SIZE] p=16 → min=4, max=12, mab_k=8 (frac=[20%, 75%])
[ENGINE] mode=C reward=lsil_fixed_calibrated dynamic_k=True C_eff=auto(Phase B)
[SUBSET] k_init=8, range=[4, 12] of 16 features
[TIME] Redundancy matrix: 12.63s
[TIME] Build reward: 18.21s


MAB:   0%|          | 0/12 [00:00<?, ?it/s]

[TIME] MAB: 0.06s


SA:   0%|          | 0/50 [00:00<?, ?it/s]

[SA] Iter 1: k= 7 | Current=0.3677 | New=0.3677 | Best=0.3677 | Accept=True | T=0.9500
[SA] Iter 2: k= 7 | Current=0.3588 | New=0.3588 | Best=0.3677 | Accept=True | T=0.9025
[SA] Iter 3: k= 7 | Current=0.3989 | New=0.3989 | Best=0.3989 | Accept=True | T=0.8574
[SA] Iter 4: k= 7 | Current=0.4099 | New=0.4099 | Best=0.4099 | Accept=True | T=0.8145
[SA] Iter 5: k= 7 | Current=0.4282 | New=0.4282 | Best=0.4282 | Accept=True | T=0.7738
[SA] Iter 6: k= 7 | Current=0.4258 | New=0.4258 | Best=0.4282 | Accept=True | T=0.7351
[SA] Iter 7: k= 6 | Current=0.4242 | New=0.4242 | Best=0.4282 | Accept=True | T=0.6983
[SA] Iter 8: k= 7 | Current=0.4258 | New=0.4258 | Best=0.4282 | Accept=True | T=0.6634
[SA] Iter 9: k= 6 | Current=0.4341 | New=0.4341 | Best=0.4341 | Accept=True | T=0.6302
[SA] Iter 10: k= 5 | Current=0.4228 | New=0.4228 | Best=0.4341 | Accept=True | T=0.5987
[SA] Iter 11: k= 5 | Current=0.3968 | New=0.3968 | Best=0.4341 | Accept=True | T=0.5688
[SA] Iter 12: k= 5 | Current=0.4228 | New

UnboundLocalError: local variable 'pd' referenced before assignment


## 2) What was produced?

Key files inside `runs/demo-<timestamp>/`:

- `features.csv` — selected feature list

- `cluster_assignments.csv` — (HHID, cluster)

- `profiles.json` / `profiles_table.csv` — aggregated per‑cluster profiles

- `profiles.md` — short narrative

- `metrics_internal.json` — timings, best K, algorithm, etc.

- `config.json` — parameters used


In [ ]:
# cell 7 - peek features / assignments (works w/ outdir or res)
import pandas as pd

# try reading from outdir if exists, otherwise from res
if outdir is not None and (outdir / "features.csv").exists():
    features = pd.read_csv(outdir / "features.csv")
else:
    features = pd.DataFrame({"feature": res.get("best_subset") or res.get("best_subset_cols") or []})

if outdir is not None and (outdir / "cluster_assignments.csv").exists():
    assign = pd.read_csv(outdir / "cluster_assignments.csv")
else:
    # build assign from res: if id_col provided use df[id_col] else _row_id
    labels = res.get("final_labels") or res.get("final_assignments") or []
    if IDCOL is None:
        assign = pd.DataFrame({"_row_id": range(len(labels)), "cluster": labels})
    else:
        assign = pd.DataFrame({IDCOL: df[IDCOL].values, "cluster": labels})

features.head(10), assign.head(10)



## 3) Quick profile visuals (optional)

Simple histograms of a few important indicators per cluster. Feel free to modify.


In [ ]:
# cell 9 - plotting
# FIX: read labels from saved file, not from res dict
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

plots_dir = outdir / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

# Read cluster assignments from saved CSV (reliable source)
assign = pd.read_csv(outdir / "cluster_assignments.csv")

if IDCOL is not None and IDCOL in df.columns and IDCOL in assign.columns:
    df_join = df.merge(assign, on=IDCOL, how="left")
else:
    df_join = df.reset_index(drop=True).copy()
    df_join["cluster"] = assign["cluster"].values

# choose numeric cols flexibly: prefer known indicators, else auto numeric
preferred = ["PlantProteinShare","FVShare","ProcessedShare","PreparedFoodShare",
             "TobaccoShare","EnergyBurden","NonFoodBurden","FoodShare",
             "DDS14_norm","DDS13_noTob_norm","DDS12_noTobPrep_norm"]
numeric_cols = [c for c in preferred if c in df_join.columns]
if not numeric_cols:
    # fallback, pick top numeric cols excluding cluster
    numeric_cols = [c for c in df_join.select_dtypes(include=[np.number]).columns if c != "cluster"][:6]

# Plot and save (600 dpi)
for i, col in enumerate(numeric_cols[:6]):
    plt.figure(figsize=(6,4))
    for k in sorted(df_join["cluster"].dropna().unique()):
        df_k = df_join[df_join["cluster"] == k]
        if df_k.shape[0] == 0: 
            continue
        plt.hist(df_k[col].dropna(), bins=40, alpha=0.45, label=f"cluster {int(k)}")
    plt.title(col)
    plt.xlabel(col)
    plt.ylabel("count")
    plt.legend()
    out_png = plots_dir / f"hist_{i}_{col}.png"
    plt.savefig(out_png, dpi=600, bbox_inches="tight")
    plt.close()

# Save cluster pie chart
plt.figure(figsize=(6,6))
vals, counts = np.unique(df_join["cluster"].dropna(), return_counts=True)
plt.pie(counts, labels=[f"{int(v)} ({c})" for v,c in zip(vals,counts)], autopct="%1.1f%%")
plt.title("Cluster distribution")
plt.savefig(plots_dir / "cluster_pie.png", dpi=600, bbox_inches="tight")
plt.close()

print("Saved plots to:", plots_dir)



## 4) Interpreting segments

Open `profiles_table.csv` and `profiles.md` to see concise summaries. The JSON file `profiles.json` contains detailed statistics (means/medians, lifts, chi‑square/ANOVA p‑values) you can use in the dissertation narrative.
